# 01 Data Understanding

This notebook is the intake pass for the Give Me Some Credit dataset. The goal is to answer a few practical questions before touching the data:

- Do the training and testing files load correctly?
- Which column is the target?
- Which columns are usable model inputs?
- What is missing, duplicated, or structurally odd?
- Is the target imbalanced?

No files are saved from this notebook. It is only for understanding the raw inputs.

In [1]:
from pathlib import Path
import pandas as pd

## Paths

All dataset work stays inside the `datasets` folder. Raw files are read from `../raw`; later notebooks write working files to `../interim` and final model-ready files to `../processed`.

In [2]:
DATASET_DIR = Path("..").resolve()
RAW_DIR = DATASET_DIR / "raw"

TRAIN_PATH = RAW_DIR / "GiveMeSomeCredit-training.csv"
TEST_PATH = RAW_DIR / "GiveMeSomeCredit-testing.csv"

print(TRAIN_PATH)
print(TEST_PATH)

D:\Navneeth_Codes\JustifAI\datasets\raw\GiveMeSomeCredit-training.csv
D:\Navneeth_Codes\JustifAI\datasets\raw\GiveMeSomeCredit-testing.csv


## Load Raw Files

The CSVs include an unnamed index-like first column. It is not a model feature, so it is removed immediately after loading.

In [3]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

train_df = train_df.drop(columns=["Unnamed: 0"], errors="ignore")
test_df = test_df.drop(columns=["Unnamed: 0"], errors="ignore")

print(f"Training rows/columns: {train_df.shape}")
print(f"Testing rows/columns:  {test_df.shape}")

Training rows/columns: (150000, 11)
Testing rows/columns:  (101503, 11)


In [4]:
train_df.head(10)

,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
0,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
1,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
2,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
3,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
4,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0
5,0,0.213179,74,0,0.375607,3500.0,3,0,1,0,1.0
6,0,0.305682,57,0,5710.000000,NaN,8,0,3,0,0.0
7,0,0.754464,39,0,0.209940,3500.0,8,0,0,0,0.0
8,0,0.116951,27,0,46.000000,NaN,2,0,0,0,NaN
9,0,0.189169,57,0,0.606291,23684.0,9,0,4,0,2.0


In [5]:
test_df.head(10)

,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
0,NaN,0.885519,43,0,0.177513,5700.0,4,0,0,0,0.0
1,NaN,0.463295,57,0,0.527237,9141.0,15,0,4,0,2.0
2,NaN,0.043275,59,0,0.687648,5083.0,12,0,1,0,2.0
3,NaN,0.280308,38,1,0.925961,3200.0,7,0,2,0,0.0
4,NaN,1.000000,27,0,0.019917,3865.0,4,0,0,0,1.0
5,NaN,0.509791,63,0,0.342429,4140.0,4,0,0,0,1.0
6,NaN,0.587778,50,0,1048.000000,0.0,5,0,0,0,3.0
7,NaN,0.046149,79,1,0.369170,3301.0,8,0,1,0,1.0
8,NaN,0.013527,68,0,2024.000000,NaN,4,0,1,0,0.0
9,NaN,1.000000,23,98,0.000000,0.0,0,98,0,98,0.0


## Column Inventory

`SeriousDlqin2yrs` is the prediction target. The remaining columns describe utilization, age, delinquency history, debt ratio, income, credit lines, real-estate loans, and dependents.

In [6]:
target_col = "SeriousDlqin2yrs"
feature_cols = [col for col in train_df.columns if col != target_col]

print("Target column:", target_col)
print("Feature columns:")
for col in feature_cols:
    print("-", col)

Target column: SeriousDlqin2yrs
Feature columns:
- RevolvingUtilizationOfUnsecuredLines
- age
- NumberOfTime30-59DaysPastDueNotWorse
- DebtRatio
- MonthlyIncome
- NumberOfOpenCreditLinesAndLoans
- NumberOfTimes90DaysLate
- NumberRealEstateLoansOrLines
- NumberOfTime60-89DaysPastDueNotWorse
- NumberOfDependents


In [7]:
column_check = pd.DataFrame({
    "train_dtype": train_df.dtypes.astype(str),
    "test_dtype": test_df.dtypes.astype(str),
    "train_missing": train_df.isna().sum(),
    "test_missing": test_df.isna().sum(),
})

column_check

,train_dtype,test_dtype,train_missing,test_missing
SeriousDlqin2yrs,int64,float64,0,101503
RevolvingUtilizationOfUnsecuredLines,float64,float64,0,0
age,int64,int64,0,0
NumberOfTime30-59DaysPastDueNotWorse,int64,int64,0,0
DebtRatio,float64,float64,0,0
MonthlyIncome,float64,float64,29731,20103
NumberOfOpenCreditLinesAndLoans,int64,int64,0,0
NumberOfTimes90DaysLate,int64,int64,0,0
NumberRealEstateLoansOrLines,int64,int64,0,0
NumberOfTime60-89DaysPastDueNotWorse,int64,int64,0,0


## Dataset Structure

This section checks types, summary statistics, and duplicates. The purpose is not to fix anything yet, only to spot what needs dedicated handling later.

In [8]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 11 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   SeriousDlqin2yrs                      150000 non-null  int64  
 1   RevolvingUtilizationOfUnsecuredLines  150000 non-null  float64
 2   age                                   150000 non-null  int64  
 3   NumberOfTime30-59DaysPastDueNotWorse  150000 non-null  int64  
 4   DebtRatio                             150000 non-null  float64
 5   MonthlyIncome                         120269 non-null  float64
 6   NumberOfOpenCreditLinesAndLoans       150000 non-null  int64  
 7   NumberOfTimes90DaysLate               150000 non-null  int64  
 8   NumberRealEstateLoansOrLines          150000 non-null  int64  
 9   NumberOfTime60-89DaysPastDueNotWorse  150000 non-null  int64  
 10  NumberOfDependents                    146076 non-null  float64
dtypes: float64(

In [9]:
test_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 101503 entries, 0 to 101502
Data columns (total 11 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   SeriousDlqin2yrs                      0 non-null       float64
 1   RevolvingUtilizationOfUnsecuredLines  101503 non-null  float64
 2   age                                   101503 non-null  int64  
 3   NumberOfTime30-59DaysPastDueNotWorse  101503 non-null  int64  
 4   DebtRatio                             101503 non-null  float64
 5   MonthlyIncome                         81400 non-null   float64
 6   NumberOfOpenCreditLinesAndLoans       101503 non-null  int64  
 7   NumberOfTimes90DaysLate               101503 non-null  int64  
 8   NumberRealEstateLoansOrLines          101503 non-null  int64  
 9   NumberOfTime60-89DaysPastDueNotWorse  101503 non-null  int64  
 10  NumberOfDependents                    98877 non-null   float64
dtypes: float64(

In [10]:
train_df.describe().T

,count,mean,std,min,25%,50%,75%,max
SeriousDlqin2yrs,150000.0,0.066840,0.249746,0.0,0.000000,0.000000,0.000000,1.0
RevolvingUtilizationOfUnsecuredLines,150000.0,6.048438,249.755371,0.0,0.029867,0.154181,0.559046,50708.0
age,150000.0,52.295207,14.771866,0.0,41.000000,52.000000,63.000000,109.0
NumberOfTime30-59DaysPastDueNotWorse,150000.0,0.421033,4.192781,0.0,0.000000,0.000000,0.000000,98.0
DebtRatio,150000.0,353.005076,2037.818523,0.0,0.175074,0.366508,0.868254,329664.0
MonthlyIncome,120269.0,6670.221237,14384.674215,0.0,3400.000000,5400.000000,8249.000000,3008750.0
NumberOfOpenCreditLinesAndLoans,150000.0,8.452760,5.145951,0.0,5.000000,8.000000,11.000000,58.0
NumberOfTimes90DaysLate,150000.0,0.265973,4.169304,0.0,0.000000,0.000000,0.000000,98.0
NumberRealEstateLoansOrLines,150000.0,1.018240,1.129771,0.0,0.000000,1.000000,2.000000,54.0
NumberOfTime60-89DaysPastDueNotWorse,150000.0,0.240387,4.155179,0.0,0.000000,0.000000,0.000000,98.0


In [11]:
test_df.describe().T

,count,mean,std,min,25%,50%,75%,max
SeriousDlqin2yrs,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
RevolvingUtilizationOfUnsecuredLines,101503.0,5.310000,196.156039,0.0,0.030131,0.152586,0.564225,21821.0
age,101503.0,52.405436,14.779756,21.0,41.000000,52.000000,63.000000,104.0
NumberOfTime30-59DaysPastDueNotWorse,101503.0,0.453770,4.538487,0.0,0.000000,0.000000,0.000000,98.0
DebtRatio,101503.0,344.475020,1632.595231,0.0,0.173423,0.364260,0.851619,268326.0
MonthlyIncome,81400.0,6855.035590,36508.600375,0.0,3408.000000,5400.000000,8200.000000,7727000.0
NumberOfOpenCreditLinesAndLoans,101503.0,8.453514,5.144100,0.0,5.000000,8.000000,11.000000,85.0
NumberOfTimes90DaysLate,101503.0,0.296691,4.515859,0.0,0.000000,0.000000,0.000000,98.0
NumberRealEstateLoansOrLines,101503.0,1.013074,1.110253,0.0,0.000000,1.000000,2.000000,37.0
NumberOfTime60-89DaysPastDueNotWorse,101503.0,0.270317,4.503578,0.0,0.000000,0.000000,0.000000,98.0


In [12]:
print(f"Duplicate rows in training data: {train_df.duplicated().sum()}")
print(f"Duplicate rows in testing data:  {test_df.duplicated().sum()}")

Duplicate rows in training data: 609
Duplicate rows in testing data:  328


## Missing Values

The important missing-value columns are expected to be `MonthlyIncome` and `NumberOfDependents`. They should be handled in later notebooks with model-based imputation, not global mean filling.

In [13]:
missing_summary = pd.DataFrame({
    "train_missing_count": train_df.isna().sum(),
    "train_missing_percent": train_df.isna().mean() * 100,
    "test_missing_count": test_df.isna().sum(),
    "test_missing_percent": test_df.isna().mean() * 100,
}).sort_values("train_missing_percent", ascending=False)

missing_summary

,train_missing_count,train_missing_percent,test_missing_count,test_missing_percent
MonthlyIncome,29731,19.820667,20103,19.805326
NumberOfDependents,3924,2.616000,2626,2.587116
SeriousDlqin2yrs,0,0.000000,101503,100.000000
age,0,0.000000,0,0.000000
RevolvingUtilizationOfUnsecuredLines,0,0.000000,0,0.000000
DebtRatio,0,0.000000,0,0.000000
NumberOfTime30-59DaysPastDueNotWorse,0,0.000000,0,0.000000
NumberOfOpenCreditLinesAndLoans,0,0.000000,0,0.000000
NumberOfTimes90DaysLate,0,0.000000,0,0.000000
NumberRealEstateLoansOrLines,0,0.000000,0,0.000000


## Target Distribution

Credit-risk datasets are usually imbalanced. This check tells us whether later modeling should use stratified splitting and imbalance-aware evaluation.

In [14]:
target_distribution = train_df[target_col].value_counts(dropna=False).rename("count").to_frame()
target_distribution["percent"] = target_distribution["count"] / len(train_df) * 100
target_distribution

,count,percent
SeriousDlqin2yrs,,
0,139974,93.316
1,10026,6.684


## Takeaways

Expected next actions:

1. Analyze missingness more carefully.
2. Review outliers and suspicious extreme values.
3. Predict missing `NumberOfDependents` and `MonthlyIncome` using Random Forest models trained on the training set.
4. Build feature-engineered and model-ready datasets.